In [28]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from go_search_problem import GoProblem, GoState
from game_runner import GameRunner
import matplotlib.pyplot as plt
import pickle
from collections import deque
import itertools
from typing import List, Tuple, Dict
from open_spiel.python.rl_environment import Environment
from agents import RandomAgent, GreedyAgent, AlphaBetaAgent



print (torch.mps.is_available())


env = Environment("go", board_size=5)
state = env.reset()
    

True


# Features
Get features for network

In [2]:
def get_features(game_state: GoState):
    """
    Map a game state to a list of features.

    Some useful functions from game_state include:
        game_state.size: size of the board
        get_pieces_coordinates(player_index): get coordinates of all pieces of a player (0 or 1)
        get_pieces_array(player_index): get a 2D array of pieces of a player (0 or 1)
        
        get_board(): get a 2D array of the board with 4 channels (player 0, player 1, empty, and player to move). 4 channels means the array will be of size 4 x n x n
    
        Descriptions of these methods can be found in the GoState

    Input:
        game_state: GoState to encode into a fixed size list of features
    Output:
        features: list of features
    """

    board_size = game_state.size
    # TODO: Encode game_state into a list of features
    features = []
    board = game_state.get_board()
    black_pieces = board[0].reshape(board_size * board_size)
    white_pieces = board[1].reshape(board_size * board_size)
    player = game_state.player_to_move()

    # stone difference
    stone_diff = np.sum(black_pieces) - np.sum(white_pieces)


    #legal actions
    legal = np.zeros(26)
    for action in game_state.legal_actions():
        legal[action] = 1

    features = np.concatenate((black_pieces, white_pieces, [player], [stone_diff], legal))

    return features




sample_features = get_features(state)

AttributeError: 'TimeStep' object has no attribute 'size'

# Defining q network

In [29]:
class DQN(nn.Module):
    def __init__(self, input_size, output_size):
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.output_layer = nn.Linear(64, output_size)
        self.relu = nn.ReLU()


    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        return self.output_layer(x)

In [30]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, complete = zip(*batch)

        #convert to tensors
        return (torch.tensor(states, dtype=torch.float32),
                torch.tensor(actions, dtype=torch.int64),
                torch.tensor(rewards, dtype=torch.float32),
                torch.tensor(next_states, dtype=torch.float32),
                torch.tensor(complete, dtype=torch.float32))

    def buffer_size(self):
        return len(self.buffer)

In [31]:

gamma = 0.99
epsilon = 1
epsilon_min = 0.01
epsilon_decay = 0.995
learning_rate = 0.1
memory_size = 10000
batch_size = 64
# black is 0, white is 1
player = 0




In [40]:
def opponent_agents(episode, game_state, current_player):
     if episode < 100:
          return np.random.choice(game_state.observations["legal_actions"][current_player])
     elif episode < 500:
         agent = GreedyAgent()
     else:
        agent = AlphaBetaAgent()


     return agent.get_move(game_state, time_limit=1)

In [41]:
def train_DQN(env, num_episodes, learning_rate, board_size, memory_size, epsilon, player):
    # action_size = board_size ** 2 + 1

    # dummy_state = env.reset()
    # input_size = len(get_features(dummy_state))

    input_size = env.observation_spec()["info_state"][player]
    action_size = env.action_spec()["num_actions"]



    # train on device
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

    policy_net = DQN(input_size, action_size)
    target_net = DQN(input_size, action_size)
    policy_net = policy_net.to(device)
    target_net = target_net.to(device)

    target_net.load_state_dict(policy_net.state_dict())


    optimizer = optim.Adam(policy_net.parameters(), lr=learning_rate)

    buffer = ReplayBuffer(memory_size)
    best_reward = float('-inf')

    

    # TODO: Implement Q-Learning
    for episode in range(num_episodes):
        total_reward = 0
        time_step = env.reset()


        while not time_step.last():
            # only train on our player's turn
            current_player = time_step.observations["current_player"]
            is_opening_move = time_step.first()




            if current_player == player:
                state = time_step.observations["info_state"][player]
                legal_actions = time_step.observations["legal_actions"][player]
                state_tensor = torch.tensor(state, dtype=torch.float32).to(device)

                if is_opening_move:
                    # hardcoded the opening move to be the center of 5x5 board.
                    action = 12


                # episilon-greedy 
                if np.random.rand() < epsilon:
                    action = np.random.choice(legal_actions)
                else:
                    q_vals = policy_net(state_tensor)

                    # masking illegal actions by setting their Q-values to -inf so they won't be chosen
                    mask = torch.full((action_size,), float('-inf'))
                    mask[legal_actions] = 0

                    action = torch.argmax(q_vals + mask).item()

                next_time_step = env.step([action])
                reward = next_time_step.rewards[player]
                complete = next_time_step.last()

                next_state = torch.tensor(next_time_step.observations["info_state"][player], dtype=torch.float32)
                buffer.push(state, action, reward, next_state, complete)


                # training
                if buffer.buffer_size() >= batch_size:
                    states, actions, rewards, next_states, complete = buffer.sample(batch_size)

                    q_values = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
                    next_q_values = target_net(next_states).max(1)[0]
                    target_q_values = rewards + (gamma * next_q_values * (1 - complete))

                    loss = nn.MSELoss()(q_values, target_q_values.detach())
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                
                total_reward += reward
            
            # agents as the opponent
            else:
                legal_actions = time_step.observations["legal_actions"][current_player]
                action = opponent_agents(episode, time_step, current_player)
                next_time_step = env.step([action])
            
            time_step = next_time_step



        if episode % 50 == 0:
            target_net.load_state_dict(policy_net.state_dict())
            
            
            
        if total_reward > best_reward:
            best_reward = total_reward
            torch.save(policy_net.state_dict(), "dqn_model.pt")

    torch.save(policy_net.state_dict(), "dqn_model.pt")
    return policy_net, target_net, buffer


value_model = train_DQN(env, num_episodes=1000, learning_rate=learning_rate, board_size=5, memory_size=memory_size, epsilon=epsilon, player=player)

ValueError: only one element tensors can be converted to Python scalars